# Practical 6 -Leakage-Safe Imbalance Handling

## Objective

Handle class imbalance without damaging temporal order, client behaviour, label confidence, or evaluation validity.

This dataset is unusual because weakly labeled attacks are the majority class. The balancing procedure will therefore:

1. preserve uncertain records instead of forcing them into a class;
2. create evaluation partitions before calculating balancing parameters;
3. calculate weights using training records only;
4. avoid changing validation and test distributions;
5. avoid duplicating or synthetically interpolating causal events;
6. retain event identity and provenance.

SMOTE is not used as the primary method because interpolating between behavioural states can create feature vectors that never represented a valid event history.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


def locate_project_root(start_path):
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = locate_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.provenance import calculate_file_sha256


CLEANED_PATH = (
    PROJECT_ROOT / "data" / "processed" / "cj_cleaned.csv"
)

LABELS_PATH = (
    PROJECT_ROOT / "data" / "labels" / "cj_weak_labels.csv"
)

FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "cj_primary_features.csv"
)

LABELS_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "labels"
    / "cj_weak_labels_manifest.json"
)

FEATURES_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "cj_primary_features_manifest.json"
)

CHUNK_SIZE = 100_000

with LABELS_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    labels_manifest = json.load(file)

with FEATURES_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    features_manifest = json.load(file)

assert (
    calculate_file_sha256(LABELS_PATH)
    == labels_manifest["output_sha256"]
)

assert (
    calculate_file_sha256(FEATURES_PATH)
    == features_manifest["output_sha256"]
)

assert (
    labels_manifest["output_rows"]
    == features_manifest["output_rows"]
)

print("Input fingerprints: passed")
print(
    "Aligned records:",
    f"{features_manifest['output_rows']:,}",
)
print(
    "Feature version:",
    features_manifest["feature_version"],
)
print(
    "Label version:",
    labels_manifest["label_version"],
)

Input fingerprints: passed
Aligned records: 2,062,361
Feature version: feature-v1
Label version: weak-label-v1


## Class-imbalance audit

`uncertain` represents abstention and is not treated as a third verified class. It will be preserved for later uncertainty analysis but excluded from binary attack-versus-benign weight calculation.

In [2]:
class_distribution = pd.Series(
    labels_manifest["class_distribution"],
    name="record_count",
).rename_axis("weak_label").to_frame()

class_distribution["coverage_percent"] = (
    class_distribution["record_count"]
    / class_distribution["record_count"].sum()
    * 100
)

supervised_counts = class_distribution.loc[
    ["attack", "benign"],
    "record_count",
]

majority_class = supervised_counts.idxmax()
minority_class = supervised_counts.idxmin()

imbalance_ratio = (
    supervised_counts.max()
    / supervised_counts.min()
)

print("Majority class:", majority_class)
print("Minority class:", minority_class)
print(
    "Supervised imbalance ratio:",
    f"{imbalance_ratio:.3f}:1",
)
print(
    "Uncertain records preserved:",
    f"{class_distribution.loc['uncertain', 'record_count']:,}",
)

class_distribution

Majority class: attack
Minority class: benign
Supervised imbalance ratio: 9.554:1
Uncertain records preserved: 40,437


,record_count,coverage_percent
weak_label,,
benign,191584,9.289547
uncertain,40437,1.960714
attack,1830340,88.749739


## Label-independent chronological partition

Records are partitioned before inspecting per-partition class frequencies:

- first 70%: training;
- next 15%: validation;
- final 15%: testing.

The split uses source-arrival position rather than labels. This matches the ordering used by causal feature generation and prevents random-row leakage.

In [3]:
from collections import Counter
from itertools import zip_longest

TOTAL_ROWS = features_manifest["output_rows"]

TRAIN_END_POSITION = int(TOTAL_ROWS * 0.70)
VALIDATION_END_POSITION = int(TOTAL_ROWS * 0.85)

SPLIT_NAMES = [
    "train",
    "validation",
    "test",
]

split_record_counts = Counter()
split_label_counts = {
    split_name: Counter()
    for split_name in SPLIT_NAMES
}

split_clients = {
    split_name: set()
    for split_name in SPLIT_NAMES
}

split_time_bounds = {
    split_name: {
        "minimum": None,
        "maximum": None,
    }
    for split_name in SPLIT_NAMES
}

cleaned_reader = pd.read_csv(
    CLEANED_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "timestamp_normalized",
        "client_ip_canonical",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
        "client_ip_canonical": str,
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

label_reader = pd.read_csv(
    LABELS_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "weak_label",
    ],
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

global_position = 0
missing_chunk = object()

for chunk_number, (
    cleaned_chunk,
    label_chunk,
) in enumerate(
    zip_longest(
        cleaned_reader,
        label_reader,
        fillvalue=missing_chunk,
    ),
    start=1,
):
    if (
        cleaned_chunk is missing_chunk
        or label_chunk is missing_chunk
    ):
        raise RuntimeError(
            "Cleaned and label chunk counts differ."
        )

    cleaned_identity = cleaned_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    label_identity = label_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    if not cleaned_identity.equals(label_identity):
        raise RuntimeError(
            f"Identity mismatch in chunk {chunk_number}."
        )

    chunk_length = len(cleaned_chunk)

    positions = np.arange(
        global_position,
        global_position + chunk_length,
    )

    split_assignments = np.select(
        [
            positions < TRAIN_END_POSITION,
            positions < VALIDATION_END_POSITION,
        ],
        [
            "train",
            "validation",
        ],
        default="test",
    )

    timestamps = pd.to_datetime(
        cleaned_chunk["timestamp_normalized"],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    ).reset_index(drop=True)

    clients = cleaned_chunk[
        "client_ip_canonical"
    ].reset_index(drop=True)

    labels = label_chunk[
        "weak_label"
    ].reset_index(drop=True)

    for split_name in SPLIT_NAMES:
        split_mask = split_assignments == split_name

        if not split_mask.any():
            continue

        split_labels = labels[split_mask]
        split_timestamps = timestamps[split_mask]

        split_record_counts[split_name] += int(
            split_mask.sum()
        )

        split_label_counts[split_name].update(
            split_labels
        )

        split_clients[split_name].update(
            clients[split_mask]
        )

        current_minimum = split_timestamps.min()
        current_maximum = split_timestamps.max()

        previous_minimum = split_time_bounds[
            split_name
        ]["minimum"]

        previous_maximum = split_time_bounds[
            split_name
        ]["maximum"]

        split_time_bounds[split_name]["minimum"] = (
            current_minimum
            if previous_minimum is None
            else min(previous_minimum, current_minimum)
        )

        split_time_bounds[split_name]["maximum"] = (
            current_maximum
            if previous_maximum is None
            else max(previous_maximum, current_maximum)
        )

    global_position += chunk_length

assert global_position == TOTAL_ROWS
assert sum(split_record_counts.values()) == TOTAL_ROWS

split_distribution = pd.DataFrame.from_dict(
    split_label_counts,
    orient="index",
).fillna(0).astype("int64")

split_distribution = split_distribution[
    ["attack", "benign", "uncertain"]
]

for label_name in [
    "attack",
    "benign",
    "uncertain",
]:
    split_distribution[
        f"{label_name}_percent"
    ] = (
        split_distribution[label_name]
        / split_distribution[
            ["attack", "benign", "uncertain"]
        ].sum(axis=1)
        * 100
    )

print(
    "Split record counts:",
    dict(split_record_counts),
)

print("\nTimestamp ranges:")

for split_name in SPLIT_NAMES:
    print(
        split_name,
        split_time_bounds[split_name],
    )

print("\nClient overlap:")
print(
    "Train-validation:",
    len(
        split_clients["train"]
        & split_clients["validation"]
    ),
)
print(
    "Train-test:",
    len(
        split_clients["train"]
        & split_clients["test"]
    ),
)
print(
    "Validation-test:",
    len(
        split_clients["validation"]
        & split_clients["test"]
    ),
)

split_distribution

Split record counts: {'train': 1443652, 'validation': 309354, 'test': 309355}

Timestamp ranges:
train {'minimum': Timestamp('2023-01-08 08:07:15'), 'maximum': Timestamp('2024-02-09 05:28:32')}
validation {'minimum': Timestamp('2024-02-09 05:28:32'), 'maximum': Timestamp('2024-02-09 15:23:14')}
test {'minimum': Timestamp('2024-02-09 15:23:14'), 'maximum': Timestamp('2024-02-19 21:44:01')}

Client overlap:
Train-validation: 18
Train-test: 249
Validation-test: 7


,attack,benign,uncertain,attack_percent,benign_percent,uncertain_percent
train,1217096,188101,38455,84.306744,13.029525,2.663731
validation,309185,24,145,99.945370,0.007758,0.046872
test,304059,3459,1837,98.288051,1.118133,0.593816


## Temporal-drift diagnosis

A fixed percentage of records does not necessarily represent a comparable duration. Large scanner bursts compressed the validation partition into a short period with almost no benign support.

Validation and testing must preserve their natural distributions. Therefore, their weakness cannot be repaired through oversampling. Calendar-level class distributions are inspected before selecting defensible development and future-test periods.

In [4]:
from collections import Counter
from itertools import zip_longest

daily_label_counts = Counter()

cleaned_time_reader = pd.read_csv(
    CLEANED_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "timestamp_normalized",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

label_time_reader = pd.read_csv(
    LABELS_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "weak_label",
    ],
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

missing_chunk = object()

for chunk_number, (
    cleaned_chunk,
    label_chunk,
) in enumerate(
    zip_longest(
        cleaned_time_reader,
        label_time_reader,
        fillvalue=missing_chunk,
    ),
    start=1,
):
    if (
        cleaned_chunk is missing_chunk
        or label_chunk is missing_chunk
    ):
        raise RuntimeError(
            "Timestamp and label files have "
            "different chunk counts."
        )

    cleaned_identity = cleaned_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    label_identity = label_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    if not cleaned_identity.equals(label_identity):
        raise RuntimeError(
            f"Identity mismatch in chunk {chunk_number}."
        )

    chunk_days = pd.to_datetime(
        cleaned_chunk["timestamp_normalized"],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    ).dt.normalize()

    chunk_summary = (
        pd.DataFrame(
            {
                "day": chunk_days,
                "weak_label": label_chunk[
                    "weak_label"
                ].to_numpy(),
            }
        )
        .groupby(
            ["day", "weak_label"],
            observed=True,
        )
        .size()
    )

    daily_label_counts.update(
        {
            key: int(value)
            for key, value in chunk_summary.items()
        }
    )

daily_distribution = (
    pd.Series(daily_label_counts)
    .rename("record_count")
    .rename_axis(["day", "weak_label"])
    .unstack(fill_value=0)
    .sort_index()
)

for label_name in [
    "attack",
    "benign",
    "uncertain",
]:
    if label_name not in daily_distribution:
        daily_distribution[label_name] = 0

daily_distribution = daily_distribution[
    ["attack", "benign", "uncertain"]
]

daily_distribution["total"] = (
    daily_distribution.sum(axis=1)
)

daily_distribution["benign_percent"] = (
    daily_distribution["benign"]
    / daily_distribution["total"]
    * 100
)

monthly_distribution = (
    daily_distribution[
        ["attack", "benign", "uncertain"]
    ]
    .resample("MS")
    .sum()
)

monthly_distribution["total"] = (
    monthly_distribution.sum(axis=1)
)

monthly_distribution["benign_percent"] = (
    monthly_distribution["benign"]
    / monthly_distribution["total"]
    * 100
)

assert (
    int(monthly_distribution["total"].sum())
    == TOTAL_ROWS
)

print("Monthly class distribution:")
display(monthly_distribution)

print("Final 30 active days:")
display(daily_distribution.tail(30))

Monthly class distribution:


weak_label,attack,benign,uncertain,total,benign_percent
day,,,,,
2023-01-01,13259,13363,3390,30012,44.525523
2023-02-01,851,18423,3137,22411,82.205167
2023-03-01,1006,12685,3898,17589,72.118938
2023-04-01,472,6972,2386,9830,70.925738
2023-05-01,394,7262,2943,10599,68.515898
2023-06-01,436,10132,2449,13017,77.836675
2023-07-01,2971,5249,2563,10783,48.678475
2023-08-01,399,5194,2556,8149,63.737882
2023-09-01,109165,8675,2654,120494,7.199529


Final 30 active days:


weak_label,attack,benign,uncertain,total,benign_percent
day,,,,,
2024-01-21,12,218,90,320,68.125000
2024-01-22,7,98,231,336,29.166667
2024-01-23,7,49,147,203,24.137931
2024-01-24,20465,77,89,20631,0.373225
2024-01-25,9,86,40,135,63.703704
2024-01-26,15,63,85,163,38.650307
2024-01-27,7,97,58,162,59.876543
2024-01-28,9,117,70,196,59.693878
2024-01-29,16,107,79,202,52.970297


## Frozen calendar split

The event-percentage split is rejected because a large scanner burst compressed validation into only several hours.

The replacement split uses complete calendar periods:

- **Training:** before 1 January 2024
- **Validation:** January 2024
- **Testing:** February 2024

This provides meaningful class support during development while preserving February as a genuinely future, naturally imbalanced stress test.

The split is frozen before model training. Test records will not be resampled, and their labels will not influence training weights or model selection.

In [5]:
import importlib
import src.balancing as balancing_module

importlib.reload(balancing_module)

from src.balancing import (
    SPLIT_VERSION,
    TEST_START,
    VALIDATION_START,
    assign_calendar_split,
)

In [6]:
boundary_examples = pd.Series(
    [
        "2023-12-31 23:59:59",
        "2024-01-01 00:00:00",
        "2024-01-31 23:59:59",
        "2024-02-01 00:00:00",
    ]
)

boundary_results = assign_calendar_split(
    boundary_examples
)

assert boundary_results.tolist() == [
    "train",
    "validation",
    "validation",
    "test",
]

second_boundary_results = assign_calendar_split(
    boundary_examples
)

pd.testing.assert_series_equal(
    boundary_results,
    second_boundary_results,
)

print("Calendar boundaries: passed")
print("Deterministic assignment: passed")

pd.DataFrame(
    {
        "timestamp": boundary_examples,
        "split": boundary_results,
    }
)

Calendar boundaries: passed
Deterministic assignment: passed


,timestamp,split
0,2023-12-31 23:59:59,train
1,2024-01-01 00:00:00,validation
2,2024-01-31 23:59:59,validation
3,2024-02-01 00:00:00,test


In [7]:
daily_split_assignments = np.select(
    [
        daily_distribution.index
        < VALIDATION_START,
        daily_distribution.index
        < TEST_START,
    ],
    [
        "train",
        "validation",
    ],
    default="test",
)

calendar_split_distribution = (
    daily_distribution[
        ["attack", "benign", "uncertain"]
    ]
    .assign(split=daily_split_assignments)
    .groupby("split")
    .sum()
    .reindex(
        ["train", "validation", "test"]
    )
    .astype("int64")
)

calendar_split_distribution["total"] = (
    calendar_split_distribution[
        ["attack", "benign", "uncertain"]
    ].sum(axis=1)
)

for label_name in [
    "attack",
    "benign",
    "uncertain",
]:
    calendar_split_distribution[
        f"{label_name}_percent"
    ] = (
        calendar_split_distribution[label_name]
        / calendar_split_distribution["total"]
        * 100
    )

assert (
    int(
        calendar_split_distribution[
            "total"
        ].sum()
    )
    == TOTAL_ROWS
)

print("Frozen split version:", SPLIT_VERSION)
print(
    "Total reconciled records:",
    f"{calendar_split_distribution['total'].sum():,}",
)

calendar_split_distribution

Frozen split version: calendar-temporal-v1
Total reconciled records: 2,062,361


weak_label,attack,benign,uncertain,total,attack_percent,benign_percent,uncertain_percent
split,,,,,,,
train,130243,102811,33198,266252,48.917191,38.614170,12.468639
validation,393419,83642,3416,480477,81.880922,17.408117,0.710960
test,1306678,5131,3823,1315632,99.319415,0.390003,0.290583


## Confidence-Normalized Class Weighting

The standard balanced class weight is:

$$
w_c = \frac{N}{K\,n_c}
$$

where:

- $N$ = total number of supervised training records
- $K$ = number of supervised classes
- $n_c$ = number of training records in class $c$
- $w_c$ = weight assigned to class $c$

For each individual record, confidence is normalized within its own class:

$$
w_i
=
w_{y_i}
\left(
\frac{q_i}{\bar{q}_{y_i}}
\right)
$$

where:

- $w_i$ = final sample weight for record $i$
- $w_{y_i}$ = balanced class weight of the record's class
- $q_i$ = confidence score of record $i$
- $\bar{q}_{y_i}$ = mean confidence of all training records in the same class

Therefore:

$$
\boxed{
\text{Final Sample Weight}
=
\text{Class Balance Weight}
\times
\text{Within-Class Relative Confidence}
}
$$

This preserves equal total influence across supervised classes while allowing
higher-confidence records within each class to contribute more strongly.

Records with the `uncertain` label are excluded from supervised balancing and
receive:

$$
w_i = 0
$$

In [8]:
import importlib
import src.balancing as balancing_module

importlib.reload(balancing_module)

from src.balancing import (
    BALANCING_VERSION,
    build_confidence_normalized_weights,
    calculate_balanced_class_weights,
    calculate_within_class_confidence_means,
)

synthetic_labels = pd.Series(
    [
        "attack",
        "attack",
        "attack",
        "benign",
        "uncertain",
    ]
)

synthetic_confidence = pd.Series(
    [
        0.85,
        0.95,
        0.99,
        0.55,
        0.00,
    ]
)

synthetic_class_weights = (
    calculate_balanced_class_weights(
        synthetic_labels
    )
)

synthetic_confidence_means = (
    calculate_within_class_confidence_means(
        synthetic_labels,
        synthetic_confidence,
    )
)

synthetic_weights = (
    build_confidence_normalized_weights(
        synthetic_labels,
        synthetic_confidence,
        synthetic_class_weights,
        synthetic_confidence_means,
    )
)

synthetic_result = pd.DataFrame(
    {
        "weak_label": synthetic_labels,
        "label_confidence": synthetic_confidence,
        "sample_weight": synthetic_weights,
    }
)

synthetic_weight_mass = (
    synthetic_result[
        synthetic_result["weak_label"].isin(
            ["attack", "benign"]
        )
    ]
    .groupby("weak_label")["sample_weight"]
    .sum()
)

assert np.isclose(
    synthetic_class_weights["attack"],
    4 / (2 * 3),
)

assert np.isclose(
    synthetic_class_weights["benign"],
    4 / (2 * 1),
)

assert np.isclose(
    synthetic_weight_mass["attack"],
    synthetic_weight_mass["benign"],
)

assert synthetic_weights.iloc[-1] == 0

assert np.isclose(
    synthetic_weights.iloc[:-1].mean(),
    1.0,
)

print("Training-only class formula: passed")
print("Within-class confidence normalization: passed")
print("Equal class weight mass: passed")
print("Uncertain abstention weight: passed")
print("Supervised mean weight: passed")

synthetic_result

Training-only class formula: passed
Within-class confidence normalization: passed
Equal class weight mass: passed
Uncertain abstention weight: passed
Supervised mean weight: passed


,weak_label,label_confidence,sample_weight
0,attack,0.85,0.609319
1,attack,0.95,0.681004
2,attack,0.99,0.709677
3,benign,0.55,2.000000
4,uncertain,0.00,0.000000


### Effective Sample Size (ESS)

Class weighting gives different training records different levels of influence.
If a small number of records receive very large weights, the effective amount of
training information can become much smaller than the actual number of records.

Effective Sample Size is calculated as:

$$
ESS =
\frac{
\left(\sum_{i=1}^{N} w_i\right)^2
}{
\sum_{i=1}^{N} w_i^2
}
$$

where:

- $N$ = number of supervised records
- $w_i$ = sample weight of record $i$
- $ESS$ = number of equally weighted records that would provide a similar amount of information

If all records have equal weights:

$$
ESS = N
$$

If a few records receive much larger weights than the others:

$$
ESS < N
$$

Therefore, an ESS close to the actual supervised record count indicates that
the weighting scheme is not excessively concentrated on a small number of records.

Records labeled `uncertain` receive a supervised weight of zero and therefore do
not contribute to supervised ESS.

In [9]:
training_parts = []

cleaned_training_reader = pd.read_csv(
    CLEANED_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "timestamp_normalized",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

label_training_reader = pd.read_csv(
    LABELS_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "weak_label",
        "label_confidence",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
        "weak_label": str,
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

missing_chunk = object()

for chunk_number, (
    cleaned_chunk,
    label_chunk,
) in enumerate(
    zip_longest(
        cleaned_training_reader,
        label_training_reader,
        fillvalue=missing_chunk,
    ),
    start=1,
):
    if (
        cleaned_chunk is missing_chunk
        or label_chunk is missing_chunk
    ):
        raise RuntimeError(
            "Cleaned and label chunk counts differ."
        )

    cleaned_identity = cleaned_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    label_identity = label_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    if not cleaned_identity.equals(label_identity):
        raise RuntimeError(
            f"Identity mismatch in chunk {chunk_number}."
        )

    timestamps = pd.to_datetime(
        cleaned_chunk["timestamp_normalized"],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )

    training_mask = (
        timestamps < VALIDATION_START
    ).to_numpy()

    if training_mask.any():
        training_parts.append(
            label_chunk.loc[
                training_mask,
                [
                    "weak_label",
                    "label_confidence",
                ],
            ].reset_index(drop=True)
        )

training_labels = pd.concat(
    training_parts,
    ignore_index=True,
)

training_labels["label_confidence"] = pd.to_numeric(
    training_labels["label_confidence"],
    errors="raise",
)

expected_training_rows = int(
    calendar_split_distribution.loc[
        "train",
        "total",
    ]
)

assert len(training_labels) == expected_training_rows

training_class_weights = (
    calculate_balanced_class_weights(
        training_labels["weak_label"]
    )
)

training_confidence_means = (
    calculate_within_class_confidence_means(
        training_labels["weak_label"],
        training_labels["label_confidence"],
    )
)

training_labels["sample_weight"] = (
    build_confidence_normalized_weights(
        training_labels["weak_label"],
        training_labels["label_confidence"],
        training_class_weights,
        training_confidence_means,
    )
)

supervised_training = training_labels[
    training_labels["weak_label"].isin(
        ["attack", "benign"]
    )
]

assert np.isclose(
    supervised_training["sample_weight"].mean(),
    1.0,
)

assert (
    training_labels.loc[
        training_labels["weak_label"]
        == "uncertain",
        "sample_weight",
    ]
    == 0
).all()

attack_weight_mass = training_labels.loc[
    training_labels["weak_label"] == "attack",
    "sample_weight",
].sum()

benign_weight_mass = training_labels.loc[
    training_labels["weak_label"] == "benign",
    "sample_weight",
].sum()

assert np.isclose(
    attack_weight_mass,
    benign_weight_mass,
)

summary_rows = []

for label_name in [
    "attack",
    "benign",
    "uncertain",
]:
    label_subset = training_labels[
        training_labels["weak_label"]
        == label_name
    ]

    weight_sum = float(
        label_subset["sample_weight"].sum()
    )

    weight_square_sum = float(
        (
            label_subset["sample_weight"] ** 2
        ).sum()
    )

    effective_sample_size = (
        weight_sum ** 2 / weight_square_sum
        if weight_square_sum > 0
        else 0.0
    )

    summary_rows.append(
        {
            "weak_label": label_name,
            "record_count": len(label_subset),
            "mean_confidence": label_subset[
                "label_confidence"
            ].mean(),
            "mean_weight": label_subset[
                "sample_weight"
            ].mean(),
            "weight_mass": weight_sum,
            "effective_sample_size":
                effective_sample_size,
        }
    )

training_weight_summary = (
    pd.DataFrame(summary_rows)
    .set_index("weak_label")
)

print(
    "Training records:",
    f"{len(training_labels):,}",
)
print(
    "Supervised training records:",
    f"{len(supervised_training):,}",
)
print(
    "Training class weights:",
    training_class_weights,
)
print(
    "Training confidence means:",
    training_confidence_means,
)
print("Equal class weight mass: passed")
print("Supervised mean weight: passed")
print("Uncertain weight policy: passed")

training_weight_summary

Training records: 266,252
Supervised training records: 233,054
Training class weights: {'attack': 0.8946891579585851, 'benign': 1.1334098491406561}
Training confidence means: {'attack': 0.8519231743740544, 'benign': 0.5499999999999998}
Equal class weight mass: passed
Supervised mean weight: passed
Uncertain weight policy: passed


,record_count,mean_confidence,mean_weight,weight_mass,effective_sample_size
weak_label,,,,,
attack,130243,0.851923,0.894689,116527.0,130205.134872
benign,102811,0.550000,1.133410,116527.0,102811.000000
uncertain,33198,0.000000,0.000000,0.0,0.000000


### Weighting result interpretation

The frozen 2023 training partition is only mildly imbalanced:

- Attack: 130,243
- Benign: 102,811
- Uncertain: 33,198

Balanced weighting gives attacks a mean weight of approximately `0.895` and benign records `1.133`. This is a small correction rather than aggressive resampling.

The attack effective sample size remains close to its physical record count, while benign ESS is unchanged because all benign weak labels currently share confidence `0.55`.

Therefore, the weighting policy balances class influence without manufacturing events, deleting attack bursts, or substantially reducing statistical information.

## Balancing sidecar

Balancing is represented as metadata rather than a rewritten dataset.

Each event receives:

- its frozen calendar split;
- binary-supervision eligibility;
- training eligibility;
- a training sample weight.

Validation and test records receive training weight zero because they must never influence model fitting. Uncertain records also receive zero supervised weight but remain preserved for later abstention analysis.

In [10]:
from datetime import datetime, timezone
from itertools import zip_longest
from collections import Counter
import time

from src.balancing import (
    BALANCING_VERSION,
    SPLIT_VERSION,
    SUPERVISED_CLASSES,
    assign_calendar_split,
    build_confidence_normalized_weights,
)

BALANCING_DIRECTORY = (
    PROJECT_ROOT / "data" / "balancing"
)

BALANCING_OUTPUT = (
    BALANCING_DIRECTORY
    / "cj_split_weights.csv"
)

BALANCING_MANIFEST_OUTPUT = (
    BALANCING_DIRECTORY
    / "cj_split_weights_manifest.json"
)

BALANCING_CODE_PATH = (
    PROJECT_ROOT / "src" / "balancing.py"
)

BALANCING_SIDECAR_COLUMNS = [
    "event_id",
    "record_hash",
    "data_split",
    "supervised_eligible",
    "training_eligible",
    "training_sample_weight",
]

actual_cleaned_hash = calculate_file_sha256(
    CLEANED_PATH
)

actual_label_hash = calculate_file_sha256(
    LABELS_PATH
)

actual_feature_hash = calculate_file_sha256(
    FEATURES_PATH
)

assert (
    actual_cleaned_hash
    == labels_manifest["input_sha256"]
    == features_manifest["input_sha256"]
)

assert (
    actual_label_hash
    == labels_manifest["output_sha256"]
)

assert (
    actual_feature_hash
    == features_manifest["output_sha256"]
)

print("Source fingerprints: passed")

Source fingerprints: passed


In [11]:
def export_balancing_sidecar():
    BALANCING_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True,
    )

    partial_output = BALANCING_OUTPUT.with_name(
        BALANCING_OUTPUT.name + ".partial"
    )

    partial_manifest = (
        BALANCING_MANIFEST_OUTPUT.with_name(
            BALANCING_MANIFEST_OUTPUT.name
            + ".partial"
        )
    )

    if (
        BALANCING_OUTPUT.exists()
        or BALANCING_MANIFEST_OUTPUT.exists()
    ):
        raise FileExistsError(
            "Final balancing artifacts already exist."
        )

    existing_partials = [
        path
        for path in [
            partial_output,
            partial_manifest,
        ]
        if path.exists()
    ]

    if existing_partials:
        raise FileExistsError(
            f"Incomplete balancing export exists: "
            f"{existing_partials}"
        )

    cleaned_reader = pd.read_csv(
        CLEANED_PATH,
        usecols=[
            "event_id",
            "record_hash",
            "timestamp_normalized",
            "client_ip_canonical",
        ],
        dtype={
            "event_id": str,
            "record_hash": str,
            "client_ip_canonical": str,
        },
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    )

    label_reader = pd.read_csv(
        LABELS_PATH,
        usecols=[
            "event_id",
            "record_hash",
            "weak_label",
            "label_confidence",
        ],
        dtype={
            "event_id": str,
            "record_hash": str,
            "weak_label": str,
        },
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    )

    feature_identity_reader = pd.read_csv(
        FEATURES_PATH,
        usecols=[
            "event_id",
            "record_hash",
        ],
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    )

    split_counts = Counter()
    split_label_counts = Counter()
    training_weight_mass = Counter()

    split_clients = {
        split_name: set()
        for split_name in [
            "train",
            "validation",
            "test",
        ]
    }

    split_time_bounds = {
        split_name: {
            "minimum": None,
            "maximum": None,
        }
        for split_name in [
            "train",
            "validation",
            "test",
        ]
    }

    training_weight_sum = 0.0
    training_weight_square_sum = 0.0
    training_weight_minimum = None
    training_weight_maximum = None
    training_eligible_count = 0

    written_rows = 0
    chunk_count = 0

    missing_chunk = object()
    started_at = time.perf_counter()

    for chunk_number, chunks in enumerate(
        zip_longest(
            cleaned_reader,
            label_reader,
            feature_identity_reader,
            fillvalue=missing_chunk,
        ),
        start=1,
    ):
        cleaned_chunk, label_chunk, feature_identity = chunks

        if any(
            chunk is missing_chunk
            for chunk in chunks
        ):
            raise RuntimeError(
                "Source artifact chunk counts differ."
            )

        cleaned_identity = cleaned_chunk[
            ["event_id", "record_hash"]
        ].reset_index(drop=True)

        label_identity = label_chunk[
            ["event_id", "record_hash"]
        ].reset_index(drop=True)

        feature_identity = feature_identity[
            ["event_id", "record_hash"]
        ].reset_index(drop=True)

        if not cleaned_identity.equals(label_identity):
            raise RuntimeError(
                f"Cleaned-label mismatch in chunk "
                f"{chunk_number}."
            )

        if not cleaned_identity.equals(feature_identity):
            raise RuntimeError(
                f"Cleaned-feature mismatch in chunk "
                f"{chunk_number}."
            )

        parsed_timestamps = pd.to_datetime(
            cleaned_chunk[
                "timestamp_normalized"
            ].reset_index(drop=True),
            format="%Y-%m-%d %H:%M:%S",
            errors="raise",
        )

        data_split = assign_calendar_split(
            parsed_timestamps
        ).reset_index(drop=True)

        labels = label_chunk[
            "weak_label"
        ].reset_index(drop=True)

        confidence = pd.to_numeric(
            label_chunk[
                "label_confidence"
            ].reset_index(drop=True),
            errors="raise",
        )

        supervised_eligible = labels.isin(
            SUPERVISED_CLASSES
        )

        training_eligible = (
            data_split.eq("train")
            & supervised_eligible
        )

        sample_weights = (
            build_confidence_normalized_weights(
                labels,
                confidence,
                training_class_weights,
                training_confidence_means,
            )
        )

        sample_weights = sample_weights.where(
            training_eligible,
            0.0,
        )

        if (
            sample_weights[
                ~training_eligible
            ] != 0
        ).any():
            raise RuntimeError(
                "Non-training record received "
                "training weight."
            )

        output_chunk = pd.DataFrame(
            {
                "event_id":
                    cleaned_identity["event_id"],
                "record_hash":
                    cleaned_identity["record_hash"],
                "data_split":
                    data_split,
                "supervised_eligible":
                    supervised_eligible.astype("uint8"),
                "training_eligible":
                    training_eligible.astype("uint8"),
                "training_sample_weight":
                    sample_weights.astype("float64"),
            }
        )

        if (
            list(output_chunk.columns)
            != BALANCING_SIDECAR_COLUMNS
        ):
            raise RuntimeError(
                "Balancing sidecar schema mismatch."
            )

        if output_chunk.isna().any().any():
            raise RuntimeError(
                f"Missing balancing value in chunk "
                f"{chunk_number}."
            )

        output_chunk.to_csv(
            partial_output,
            mode="w" if chunk_number == 1 else "a",
            header=(chunk_number == 1),
            index=False,
            float_format="%.12g",
            lineterminator="\n",
        )

        count_frame = pd.DataFrame(
            {
                "data_split": data_split,
                "weak_label": labels,
            }
        )

        chunk_split_label_counts = (
            count_frame
            .groupby(
                ["data_split", "weak_label"],
                observed=True,
            )
            .size()
        )

        for key, value in (
            chunk_split_label_counts.items()
        ):
            split_label_counts[key] += int(value)

        chunk_split_counts = (
            data_split.value_counts()
        )

        for split_name, value in (
            chunk_split_counts.items()
        ):
            split_counts[str(split_name)] += int(
                value
            )

        clients = cleaned_chunk[
            "client_ip_canonical"
        ].reset_index(drop=True)

        for split_name in [
            "train",
            "validation",
            "test",
        ]:
            split_mask = data_split.eq(split_name)

            if not split_mask.any():
                continue

            split_clients[split_name].update(
                clients[split_mask]
            )

            split_timestamps = (
                parsed_timestamps[split_mask]
            )

            current_minimum = (
                split_timestamps.min()
            )

            current_maximum = (
                split_timestamps.max()
            )

            previous_minimum = (
                split_time_bounds[
                    split_name
                ]["minimum"]
            )

            previous_maximum = (
                split_time_bounds[
                    split_name
                ]["maximum"]
            )

            split_time_bounds[
                split_name
            ]["minimum"] = (
                current_minimum
                if previous_minimum is None
                else min(
                    previous_minimum,
                    current_minimum,
                )
            )

            split_time_bounds[
                split_name
            ]["maximum"] = (
                current_maximum
                if previous_maximum is None
                else max(
                    previous_maximum,
                    current_maximum,
                )
            )

        eligible_weights = sample_weights[
            training_eligible
        ]

        if len(eligible_weights):
            chunk_weight_minimum = float(
                eligible_weights.min()
            )

            chunk_weight_maximum = float(
                eligible_weights.max()
            )

            training_weight_minimum = (
                chunk_weight_minimum
                if training_weight_minimum is None
                else min(
                    training_weight_minimum,
                    chunk_weight_minimum,
                )
            )

            training_weight_maximum = (
                chunk_weight_maximum
                if training_weight_maximum is None
                else max(
                    training_weight_maximum,
                    chunk_weight_maximum,
                )
            )

            training_weight_sum += float(
                eligible_weights.sum()
            )

            training_weight_square_sum += float(
                (eligible_weights ** 2).sum()
            )

            training_eligible_count += int(
                training_eligible.sum()
            )

            for label_name in SUPERVISED_CLASSES:
                class_mask = (
                    training_eligible
                    & labels.eq(label_name)
                )

                training_weight_mass[
                    label_name
                ] += float(
                    sample_weights[class_mask].sum()
                )

        written_rows += len(output_chunk)
        chunk_count += 1

    expected_rows = features_manifest["output_rows"]

    if written_rows != expected_rows:
        raise RuntimeError(
            "Balancing output row count mismatch."
        )

    expected_training_eligible = int(
        calendar_split_distribution.loc[
            "train",
            ["attack", "benign"],
        ].sum()
    )

    if (
        training_eligible_count
        != expected_training_eligible
    ):
        raise RuntimeError(
            "Training eligibility count mismatch."
        )

    if not np.isclose(
        training_weight_mass["attack"],
        training_weight_mass["benign"],
    ):
        raise RuntimeError(
            "Training class weight masses differ."
        )

    effective_sample_size = (
        training_weight_sum ** 2
        / training_weight_square_sum
    )

    output_hash = calculate_file_sha256(
        partial_output
    )

    balancing_code_hash = calculate_file_sha256(
        BALANCING_CODE_PATH
    )

    split_distribution_manifest = {
        split_name: {
            label_name: int(
                split_label_counts[
                    (split_name, label_name)
                ]
            )
            for label_name in [
                "attack",
                "benign",
                "uncertain",
            ]
        }
        for split_name in [
            "train",
            "validation",
            "test",
        ]
    }

    client_overlap = {
        "train_validation": len(
            split_clients["train"]
            & split_clients["validation"]
        ),
        "train_test": len(
            split_clients["train"]
            & split_clients["test"]
        ),
        "validation_test": len(
            split_clients["validation"]
            & split_clients["test"]
        ),
    }

    manifest = {
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "split_version": SPLIT_VERSION,
        "balancing_version": BALANCING_VERSION,
        "cleaned_file": CLEANED_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "label_file": LABELS_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "feature_file": FEATURES_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "output_file": BALANCING_OUTPUT.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "cleaned_sha256": actual_cleaned_hash,
        "label_sha256": actual_label_hash,
        "feature_sha256": actual_feature_hash,
        "output_sha256": output_hash,
        "balancing_code_file":
            BALANCING_CODE_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "balancing_code_sha256":
            balancing_code_hash,
        "input_rows": expected_rows,
        "output_rows": written_rows,
        "chunk_count": chunk_count,
        "chunk_size": CHUNK_SIZE,
        "sidecar_columns":
            BALANCING_SIDECAR_COLUMNS,
        "validation_start":
            VALIDATION_START.isoformat(),
        "test_start":
            TEST_START.isoformat(),
        "split_counts": {
            key: int(value)
            for key, value in split_counts.items()
        },
        "split_label_distribution":
            split_distribution_manifest,
        "split_time_bounds": {
            split_name: {
                bound_name: timestamp.isoformat()
                for bound_name, timestamp in bounds.items()
            }
            for split_name, bounds in (
                split_time_bounds.items()
            )
        },
        "split_client_counts": {
            split_name: len(client_set)
            for split_name, client_set in (
                split_clients.items()
            )
        },
        "client_overlap": client_overlap,
        "supervised_classes": list(
            SUPERVISED_CLASSES
        ),
        "uncertain_policy":
            "preserve_with_zero_training_weight",
        "validation_test_weight_policy":
            "zero_training_weight",
        "class_weights": {
            key: float(value)
            for key, value in (
                training_class_weights.items()
            )
        },
        "within_class_confidence_means": {
            key: float(value)
            for key, value in (
                training_confidence_means.items()
            )
        },
        "training_eligible_count":
            training_eligible_count,
        "training_weight_mass": {
            key: float(value)
            for key, value in (
                training_weight_mass.items()
            )
        },
        "training_weight_minimum":
            training_weight_minimum,
        "training_weight_maximum":
            training_weight_maximum,
        "training_weight_mean": (
            training_weight_sum
            / training_eligible_count
        ),
        "training_effective_sample_size":
            effective_sample_size,
        "elapsed_seconds": round(
            time.perf_counter() - started_at,
            2,
        ),
    }

    with partial_manifest.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            manifest,
            file,
            indent=2,
        )

    partial_output.replace(BALANCING_OUTPUT)
    partial_manifest.replace(
        BALANCING_MANIFEST_OUTPUT
    )

    return manifest

In [12]:
def load_or_export_balancing_sidecar():
    output_exists = BALANCING_OUTPUT.exists()
    manifest_exists = (
        BALANCING_MANIFEST_OUTPUT.exists()
    )

    if output_exists != manifest_exists:
        raise RuntimeError(
            "Incomplete balancing artifact set."
        )

    if not output_exists:
        print("Creating balancing sidecar...")
        return export_balancing_sidecar()

    with BALANCING_MANIFEST_OUTPUT.open(
        "r",
        encoding="utf-8",
    ) as file:
        existing_manifest = json.load(file)

    expected_hashes = {
        "cleaned_sha256": actual_cleaned_hash,
        "label_sha256": actual_label_hash,
        "feature_sha256": actual_feature_hash,
        "balancing_code_sha256":
            calculate_file_sha256(
                BALANCING_CODE_PATH
            ),
        "output_sha256":
            calculate_file_sha256(
                BALANCING_OUTPUT
            ),
    }

    for field_name, actual_hash in (
        expected_hashes.items()
    ):
        if (
            existing_manifest[field_name]
            != actual_hash
        ):
            raise RuntimeError(
                f"Fingerprint mismatch: {field_name}"
            )

    if (
        existing_manifest["split_version"]
        != SPLIT_VERSION
    ):
        raise RuntimeError("Split version mismatch.")

    if (
        existing_manifest["balancing_version"]
        != BALANCING_VERSION
    ):
        raise RuntimeError(
            "Balancing version mismatch."
        )

    print("Reusing verified balancing sidecar.")
    return existing_manifest


balancing_export_result = (
    load_or_export_balancing_sidecar()
)

balancing_export_result

Reusing verified balancing sidecar.


{'created_at_utc': '2026-09-04T08:20:12.195399+00:00',
 'split_version': 'calendar-temporal-v1',
 'balancing_version': 'class-confidence-v1',
 'cleaned_file': 'data/processed/cj_cleaned.csv',
 'label_file': 'data/labels/cj_weak_labels.csv',
 'feature_file': 'data/features/cj_primary_features.csv',
 'output_file': 'data/balancing/cj_split_weights.csv',
 'cleaned_sha256': 'c022cee1cb2de32f7bb1db1d38d5d1dad1079b2fb590f854f9aead7fa54995f2',
 'label_sha256': '17e6c785c07056fd973957192653c0fb97706477477c2ac009c9c5691857e444',
 'feature_sha256': '0dca75a12f5fe290dcc3e398951b3e9dd1ecc823f380a64024dfc39801ad6b8e',
 'output_sha256': '1eff4ca44b66dd4947e872af34106f9dd9c174a8dd7e0cb1b6833b6e560ad97c',
 'balancing_code_file': 'src/balancing.py',
 'balancing_code_sha256': 'b90a6e77dfb6dcb5cb7f3c1d6abeadd79cf9447c70b91d50077848afeda22ea5',
 'input_rows': 2062361,
 'output_rows': 2062361,
 'chunk_count': 21,
 'chunk_size': 100000,
 'sidecar_columns': ['event_id',
  'record_hash',
  'data_split',
  'su

## Independent balancing verification

The balancing sidecar is reconstructed independently from timestamps, labels, confidence values, and frozen training parameters.

This verifies that:

- every artifact remains identity-aligned;
- split assignment follows the frozen calendar policy;
- only supervised training records receive weight;
- validation, test, and uncertain records have zero training weight;
- serialized weights reproduce the documented formula.

In [ ]:
balancing_manifest = balancing_export_result

assert (
    calculate_file_sha256(BALANCING_OUTPUT)
    == balancing_manifest["output_sha256"]
)

assert (
    calculate_file_sha256(BALANCING_CODE_PATH)
    == balancing_manifest[
        "balancing_code_sha256"
    ]
)

balancing_header = pd.read_csv(
    BALANCING_OUTPUT,
    nrows=0,
).columns.tolist()

assert balancing_header == BALANCING_SIDECAR_COLUMNS

cleaned_verify_reader = pd.read_csv(
    CLEANED_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "timestamp_normalized",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

label_verify_reader = pd.read_csv(
    LABELS_PATH,
    usecols=[
        "event_id",
        "record_hash",
        "weak_label",
        "label_confidence",
    ],
    dtype={
        "event_id": str,
        "record_hash": str,
        "weak_label": str,
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

feature_verify_reader = pd.read_csv(
    FEATURES_PATH,
    usecols=[
        "event_id",
        "record_hash",
    ],
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

balancing_verify_reader = pd.read_csv(
    BALANCING_OUTPUT,
    dtype={
        "event_id": str,
        "record_hash": str,
        "data_split": "string",
    },
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

verified_rows = 0
verified_chunks = 0
verified_training_count = 0

verified_split_counts = Counter()
verified_split_label_counts = Counter()
verified_weight_mass = Counter()

verified_weight_sum = 0.0
verified_weight_square_sum = 0.0
verified_weight_minimum = None
verified_weight_maximum = None

missing_chunk = object()

for chunk_number, chunks in enumerate(
    zip_longest(
        cleaned_verify_reader,
        label_verify_reader,
        feature_verify_reader,
        balancing_verify_reader,
        fillvalue=missing_chunk,
    ),
    start=1,
):
    (
        cleaned_chunk,
        label_chunk,
        feature_chunk,
        balancing_chunk,
    ) = chunks

    if any(
        chunk is missing_chunk
        for chunk in chunks
    ):
        raise RuntimeError(
            "Verification chunk counts differ."
        )

    cleaned_identity = cleaned_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    label_identity = label_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    feature_identity = feature_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    balancing_identity = balancing_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    if not (
        cleaned_identity.equals(label_identity)
        and cleaned_identity.equals(feature_identity)
        and cleaned_identity.equals(
            balancing_identity
        )
    ):
        raise RuntimeError(
            f"Identity mismatch in chunk "
            f"{chunk_number}."
        )

    timestamps = pd.to_datetime(
        cleaned_chunk[
            "timestamp_normalized"
        ].reset_index(drop=True),
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )

    labels = label_chunk[
        "weak_label"
    ].reset_index(drop=True)

    confidence = pd.to_numeric(
        label_chunk[
            "label_confidence"
        ].reset_index(drop=True),
        errors="raise",
    )

    expected_split = assign_calendar_split(
        timestamps
    ).reset_index(drop=True)

    expected_supervised = labels.isin(
        SUPERVISED_CLASSES
    )

    expected_training = (
        expected_split.eq("train")
        & expected_supervised
    )

    expected_weights = (
        build_confidence_normalized_weights(
            labels,
            confidence,
            training_class_weights,
            training_confidence_means,
        )
        .where(expected_training, 0.0)
    )

    actual_split =actual_split = (
    balancing_chunk["data_split"]
    .astype("string")
    .reset_index(drop=True)
    )
    actual_supervised = pd.to_numeric(
        balancing_chunk[
            "supervised_eligible"
        ],
        errors="raise",
    ).reset_index(drop=True)

    actual_training = pd.to_numeric(
        balancing_chunk[
            "training_eligible"
        ],
        errors="raise",
    ).reset_index(drop=True)

    actual_weights = pd.to_numeric(
        balancing_chunk[
            "training_sample_weight"
        ],
        errors="raise",
    ).reset_index(drop=True)

    if not actual_split.equals(expected_split):
        raise RuntimeError(
            f"Split mismatch in chunk "
            f"{chunk_number}."
        )

    if not np.array_equal(
        actual_supervised.to_numpy(),
        expected_supervised.astype(
            "uint8"
        ).to_numpy(),
    ):
        raise RuntimeError(
            "Supervised eligibility mismatch."
        )

    if not np.array_equal(
        actual_training.to_numpy(),
        expected_training.astype(
            "uint8"
        ).to_numpy(),
    ):
        raise RuntimeError(
            "Training eligibility mismatch."
        )

    if not np.allclose(
        actual_weights.to_numpy(),
        expected_weights.to_numpy(),
        rtol=1e-10,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"Weight mismatch in chunk "
            f"{chunk_number}."
        )

    if (
        actual_weights[~expected_training] != 0
    ).any():
        raise RuntimeError(
            "Non-training record has training weight."
        )

    verified_split_counts.update(
        expected_split.tolist()
    )

    verified_split_label_counts.update(
        zip(
            expected_split.tolist(),
            labels.tolist(),
        )
    )

    eligible_weights = actual_weights[
        expected_training
    ]

    if len(eligible_weights):
        verified_training_count += len(
            eligible_weights
        )

        verified_weight_sum += float(
            eligible_weights.sum()
        )

        verified_weight_square_sum += float(
            (eligible_weights ** 2).sum()
        )

        chunk_minimum = float(
            eligible_weights.min()
        )

        chunk_maximum = float(
            eligible_weights.max()
        )

        verified_weight_minimum = (
            chunk_minimum
            if verified_weight_minimum is None
            else min(
                verified_weight_minimum,
                chunk_minimum,
            )
        )

        verified_weight_maximum = (
            chunk_maximum
            if verified_weight_maximum is None
            else max(
                verified_weight_maximum,
                chunk_maximum,
            )
        )

        for label_name in SUPERVISED_CLASSES:
            class_mask = (
                expected_training
                & labels.eq(label_name)
            )

            verified_weight_mass[
                label_name
            ] += float(
                actual_weights[class_mask].sum()
            )

    verified_rows += len(balancing_chunk)
    verified_chunks += 1

assert verified_rows == balancing_manifest["output_rows"]
assert verified_chunks == balancing_manifest["chunk_count"]

assert (
    verified_training_count
    == balancing_manifest[
        "training_eligible_count"
    ]
)

for split_name in [
    "train",
    "validation",
    "test",
]:
    assert (
        verified_split_counts[split_name]
        == balancing_manifest[
            "split_counts"
        ][split_name]
    )

    for label_name in [
        "attack",
        "benign",
        "uncertain",
    ]:
        assert (
            verified_split_label_counts[
                (split_name, label_name)
            ]
            == balancing_manifest[
                "split_label_distribution"
            ][split_name][label_name]
        )

for label_name in SUPERVISED_CLASSES:
    assert np.isclose(
        verified_weight_mass[label_name],
        balancing_manifest[
            "training_weight_mass"
        ][label_name],
    )

verified_weight_mean = (
    verified_weight_sum
    / verified_training_count
)

verified_effective_sample_size = (
    verified_weight_sum ** 2
    / verified_weight_square_sum
)

assert np.isclose(
    verified_weight_minimum,
    balancing_manifest[
        "training_weight_minimum"
    ],
)

assert np.isclose(
    verified_weight_maximum,
    balancing_manifest[
        "training_weight_maximum"
    ],
)

assert np.isclose(
    verified_weight_mean,
    balancing_manifest[
        "training_weight_mean"
    ],
)

assert np.isclose(
    verified_effective_sample_size,
    balancing_manifest[
        "training_effective_sample_size"
    ],
)

partial_files = [
    BALANCING_OUTPUT.with_name(
        BALANCING_OUTPUT.name + ".partial"
    ),
    BALANCING_MANIFEST_OUTPUT.with_name(
        BALANCING_MANIFEST_OUTPUT.name
        + ".partial"
    ),
]

assert not any(
    path.exists()
    for path in partial_files
)

print("Verified rows:", f"{verified_rows:,}")
print("Verified chunks:", verified_chunks)
print("Four-artifact identity alignment: passed")
print("Calendar split reconstruction: passed")
print("Eligibility reconstruction: passed")
print("Training-weight reconstruction: passed")
print("Validation/test isolation: passed")
print("Manifest reconciliation: passed")
print(
    "Training effective sample size:",
    round(verified_effective_sample_size, 2),
)
print(
    "Balancing file size (MiB):",
    round(
        BALANCING_OUTPUT.stat().st_size
        / (1024 ** 2),
        2,
    ),
)
print("Partial files remaining: 0")

Verified rows: 2,062,361
Verified chunks: 21
Four-artifact identity alignment: passed
Calendar split reconstruction: passed
Eligibility reconstruction: passed
Training-weight reconstruction: passed
Validation/test isolation: passed
Manifest reconciliation: passed
Training effective sample size: 229795.59
Balancing file size (MiB): 136.51
Partial files remaining: 0


: 

## Conclusion

Practical 6 created a leakage-safe imbalance-handling policy for all 2,062,361 events.

### Frozen temporal partitions

- Training — 2023: 266,252 records
- Validation — January 2024: 480,477 records
- Test — February 2024: 1,315,632 records

The initial percentage-based temporal split was rejected because a concentrated scanner burst left validation with only 24 benign records. Complete calendar periods provide stronger validation support while retaining February as a naturally imbalanced future stress test.

### Training-only balancing

Only the 233,054 supervised training records participate in weight estimation:

- Attack weight: approximately 0.895
- Benign weight: approximately 1.133
- Uncertain weight: 0
- Validation and test training weight: 0

Confidence was normalized within each class before being combined with class weight. This prevents the systematically lower benign confidence policy from cancelling minority-class balancing.

The final training effective sample size is approximately 229,796, showing that weighting preserves most statistical information.

No rows were deleted, duplicated, randomly undersampled, oversampled, or synthetically interpolated. The balancing sidecar preserves event identity and separates training metadata from features and labels.